# Tahap 04 — Manual Coding: Open Coding, Axial Coding, dan Selective Coding

## Judul Project

**Analisis Komputasional Topik Pidato Presiden Prabowo pada Forum Nasional dan Internasional Menggunakan Manual Coding dan BERTopic**

## Tujuan Notebook

Notebook ini digunakan untuk menyiapkan dan membantu proses **Manual Coding** terhadap hasil chunking pidato.

Tahap ini mencakup:

1. Membaca output Tahap 03.
2. Memvalidasi struktur data chunk.
3. Membuat codebook awal untuk manual coding.
4. Membuat template manual coding per chunk.
5. Memberikan *rule-assisted suggestion* berbasis keyword secara transparan.
6. Menyimpan template untuk direview secara manual.
7. Menyediakan mekanisme opsional untuk membaca hasil coding manual yang sudah direview.
8. Membuat ringkasan kualitas dan distribusi hasil coding.

## Input Utama

```text
data/processed/speech_chunks_master.csv
data/processed/speech_chunks_for_bertopic.csv
```

## Output Utama

```text
data/interim/manual_coding_codebook.csv
data/interim/manual_coding_template.csv
data/interim/manual_coding_rule_assisted_draft.csv
reports/tables/manual_coding_suggestion_summary.csv
reports/tables/manual_coding_quality_report.csv
reports/tables/stage04_output_manifest.json
```

## Catatan

Kolom `suggested_*` pada notebook ini **bukan hasil final manual coding**.  
Kolom tersebut hanya bantuan awal berbasis aturan keyword agar proses coding lebih cepat dan konsisten.

Hasil final tetap harus direview secara manual oleh peneliti dengan mengisi kolom:

- `manual_open_code_1`
- `manual_open_code_2`
- `manual_open_code_3`
- `manual_axial_category`
- `manual_selective_theme`
- `coding_status`
- `coder_notes`

## 1. Import Library

Notebook ini menggunakan library standar Python dan Pandas agar mudah dijalankan di Jupyter Notebook lokal.

In [16]:
# ============================================================
# Import Library
# ============================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json
import re
import sys

import pandas as pd

print("Library berhasil di-import.")
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")

Library berhasil di-import.
Python version: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
Pandas version: 2.2.2


## 2. Setup Path Project

Cell ini mendeteksi root folder project berdasarkan keberadaan file hasil Tahap 03.

File yang dicari:

```text
data/processed/speech_chunks_master.csv
```

In [17]:
# ============================================================
# Setup Path Project
# ============================================================

def find_project_root(start_path=None):
    """
    Mendeteksi root folder project berdasarkan keberadaan file hasil Tahap 03.
    """
    if start_path is None:
        start_path = Path.cwd().resolve()
    else:
        start_path = Path(start_path).resolve()

    candidate_paths = [start_path] + list(start_path.parents)

    for candidate in candidate_paths:
        expected_input = candidate / "data" / "processed" / "speech_chunks_master.csv"
        if expected_input.exists():
            return candidate

    return start_path


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORT_TABLE_DIR = REPORTS_DIR / "tables"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CHUNKS_MASTER = PROCESSED_DIR / "speech_chunks_master.csv"
INPUT_CHUNKS_BERTOPIC = PROCESSED_DIR / "speech_chunks_for_bertopic.csv"

print("Project path berhasil disiapkan.")
print(f"Current working directory : {Path.cwd().resolve()}")
print(f"PROJECT_ROOT              : {PROJECT_ROOT}")
print(f"INPUT_CHUNKS_MASTER       : {INPUT_CHUNKS_MASTER}")
print(f"INPUT_CHUNKS_BERTOPIC     : {INPUT_CHUNKS_BERTOPIC}")
print(f"INTERIM_DIR               : {INTERIM_DIR}")
print(f"REPORT_TABLE_DIR          : {REPORT_TABLE_DIR}")

Project path berhasil disiapkan.
Current working directory : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\notebooks
PROJECT_ROOT              : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech
INPUT_CHUNKS_MASTER       : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\speech_chunks_master.csv
INPUT_CHUNKS_BERTOPIC     : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\speech_chunks_for_bertopic.csv
INTERIM_DIR               : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\interim
REPORT_TABLE_DIR          : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables


## 3. Preflight Check Input Tahap 03

Notebook akan berhenti jika file input utama belum tersedia.

Pastikan Tahap 03 sudah menghasilkan file:

```text
data/processed/speech_chunks_master.csv
```

In [18]:
# ============================================================
# Preflight Check Input
# ============================================================

if not INPUT_CHUNKS_MASTER.exists():
    raise FileNotFoundError(
        f"File input Tahap 03 tidak ditemukan: {INPUT_CHUNKS_MASTER}\n"
        "Pastikan Tahap 03 sudah dijalankan dan file speech_chunks_master.csv "
        "tersimpan di data/processed/."
    )

if not INPUT_CHUNKS_BERTOPIC.exists():
    print(
        "Peringatan: speech_chunks_for_bertopic.csv tidak ditemukan. "
        "Tahap 04 tetap dapat berjalan menggunakan speech_chunks_master.csv."
    )
else:
    print("File speech_chunks_for_bertopic.csv ditemukan.")

print("File input utama Tahap 03 ditemukan.")
print(f"Ukuran file speech_chunks_master.csv: {INPUT_CHUNKS_MASTER.stat().st_size:,} bytes")

File speech_chunks_for_bertopic.csv ditemukan.
File input utama Tahap 03 ditemukan.
Ukuran file speech_chunks_master.csv: 171,723 bytes


## 4. Membaca Dataset Chunk

Dataset `speech_chunks_master.csv` digunakan karena memiliki metadata yang lebih lengkap dibandingkan file khusus BERTopic.

In [19]:
# ============================================================
# Load Dataset Chunk
# ============================================================

chunks_df = pd.read_csv(INPUT_CHUNKS_MASTER)

print("Dataset chunk berhasil dibaca.")
print(f"Shape dataset: {chunks_df.shape}")

display(chunks_df.head())

print("Daftar kolom:")
for col in chunks_df.columns:
    print(f"- {col}")

Dataset chunk berhasil dibaca.
Shape dataset: (74, 25)


,chunk_id,speech_id,file_name,speech_title_from_filename,forum_scope_inferred,event_date,language_estimate,source_url,source_domain,source_validation_status,...,sentence_end_index,sentence_count,chunk_hash,chunk_quality_flags,source_text_column,chunking_strategy,chunking_config_json,word_count_clean,char_count_clean,processed_stage03_at
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,15,16,e3f58835ac64de59a5c04e132c1f112eaecdfc171fa336...,OK,text_for_bertopic,sentence_based_chunking_with_small_overlap,"{""strategy"": ""sentence_based_chunking_with_sma...",254,1642,2026-06-11T21:58:49
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,19,5,768d661bdd3e196267566917363f69a38d216b4d38c046...,LOW_WORD_COUNT_LT_80,text_for_bertopic,sentence_based_chunking_with_small_overlap,"{""strategy"": ""sentence_based_chunking_with_sma...",254,1642,2026-06-11T21:58:49
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,19,20,28d329c7546f5cb19a3aa4329ad7ce62ed7a20ff399728...,OK,text_for_bertopic,sentence_based_chunking_with_small_overlap,"{""strategy"": ""sentence_based_chunking_with_sma...",3388,24220,2026-06-11T21:58:49
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,34,16,a210025e9b6d730f70a09850905e7a0098876774ed3b5c...,OK,text_for_bertopic,sentence_based_chunking_with_small_overlap,"{""strategy"": ""sentence_based_chunking_with_sma...",3388,24220,2026-06-11T21:58:49
4,SPCH_002_PANEN_RAYA_CHK_003,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,49,16,b550fdce49dc7360e42df804c8d8aa5960a4d7f3af7ab2...,OK,text_for_bertopic,sentence_based_chunking_with_small_overlap,"{""strategy"": ""sentence_based_chunking_with_sma...",3388,24220,2026-06-11T21:58:49


Daftar kolom:
- chunk_id
- speech_id
- file_name
- speech_title_from_filename
- forum_scope_inferred
- event_date
- language_estimate
- source_url
- source_domain
- source_validation_status
- chunk_order
- chunk_text
- chunk_word_count
- chunk_char_count
- sentence_start_index
- sentence_end_index
- sentence_count
- chunk_hash
- chunk_quality_flags
- source_text_column
- chunking_strategy
- chunking_config_json
- word_count_clean
- char_count_clean
- processed_stage03_at


## 5. Validasi Kolom Wajib

Kolom wajib untuk Manual Coding:

- `chunk_id`
- `speech_id`
- `file_name`
- `chunk_order`
- `chunk_text`
- `chunk_word_count`

Kolom metadata tambahan akan digunakan apabila tersedia.

In [20]:
# ============================================================
# Helper Validasi Kolom
# ============================================================

def require_columns(df, required_columns, df_name="DataFrame"):
    """
    Memastikan DataFrame memiliki kolom yang dibutuhkan.
    """
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(
            f"{df_name} tidak memiliki kolom wajib: {missing_columns}. "
            f"Kolom tersedia: {list(df.columns)}"
        )


def safe_get(row, column_name, default_value=None):
    """
    Mengambil nilai kolom secara aman dari row Pandas.
    """
    if column_name not in row.index:
        return default_value

    value = row[column_name]

    if pd.isna(value):
        return default_value

    return value


REQUIRED_CHUNK_COLUMNS = [
    "chunk_id",
    "speech_id",
    "file_name",
    "chunk_order",
    "chunk_text",
    "chunk_word_count"
]

require_columns(chunks_df, REQUIRED_CHUNK_COLUMNS, "chunks_df")

if chunks_df["chunk_id"].isna().any():
    raise ValueError("Terdapat chunk_id kosong pada chunks_df.")

if chunks_df["chunk_id"].duplicated().any():
    duplicated_chunk_ids = chunks_df.loc[chunks_df["chunk_id"].duplicated(), "chunk_id"].tolist()
    raise ValueError(f"Terdapat chunk_id duplikat: {duplicated_chunk_ids}")

if chunks_df["chunk_text"].isna().any():
    raise ValueError("Terdapat chunk_text kosong pada chunks_df.")

if (chunks_df["chunk_word_count"] <= 0).any():
    raise ValueError("Terdapat chunk_word_count yang tidak valid.")

print("Validasi kolom wajib berhasil.")
print(f"Jumlah chunk: {len(chunks_df)}")
print(f"Jumlah speech_id unik: {chunks_df['speech_id'].nunique()}")

Validasi kolom wajib berhasil.
Jumlah chunk: 74
Jumlah speech_id unik: 6


## 6. Membuat Codebook Awal Manual Coding

Codebook awal disusun berdasarkan tema konseptual yang relevan dengan isi pidato.

Struktur codebook:

1. **Open Code**: kode awal yang dekat dengan isi teks.
2. **Axial Category**: pengelompokan beberapa open code yang saling berhubungan.
3. **Selective Theme**: tema besar yang menjadi hasil abstraksi utama.

Catatan:

Codebook ini bersifat awal dan boleh direvisi setelah pembacaan manual terhadap chunk.

In [21]:
# ============================================================
# Cell 06 — Membuat Codebook Awal
# ============================================================

codebook_records = [
    {
        "open_code_id": "OC001",
        "open_code_label": "Kedaulatan pangan dan swasembada",
        "axial_category": "Ketahanan nasional dan kedaulatan strategis",
        "selective_theme": "Kedaulatan dan kemandirian nasional",
        "keywords": [
            "swasembada", "pangan", "beras", "jagung", "petani", "pertanian",
            "food", "rice", "grain", "farmer", "farmers", "self-sufficient", "sufficiency"
        ],
        "description": "Mengacu pada narasi kemandirian pangan, produksi beras, petani, dan swasembada."
    },
    {
        "open_code_id": "OC002",
        "open_code_label": "Kedaulatan energi dan pengelolaan sumber daya",
        "axial_category": "Ketahanan nasional dan kedaulatan strategis",
        "selective_theme": "Kedaulatan dan kemandirian nasional",
        "keywords": [
            "energi", "pertamina", "biodiesel", "solar", "bbm", "esdm", "kilang",
            "energy", "fuel", "oil", "gas", "renewable", "renewables", "refinery"
        ],
        "description": "Mengacu pada kemandirian energi, Pertamina, BBM, energi terbarukan, dan pengelolaan sumber daya."
    },
    {
        "open_code_id": "OC003",
        "open_code_label": "Pemerataan kesejahteraan dan pengentasan kemiskinan",
        "axial_category": "Kesejahteraan sosial dan pemerataan pembangunan",
        "selective_theme": "Pembangunan manusia dan keadilan sosial",
        "keywords": [
            "kemiskinan", "miskin", "kesejahteraan", "rakyat", "kelaparan",
            "poverty", "poor", "hunger", "livelihood", "livelihoods", "weak", "weakest"
        ],
        "description": "Mengacu pada upaya peningkatan kesejahteraan rakyat dan pengurangan kemiskinan."
    },
    {
        "open_code_id": "OC004",
        "open_code_label": "Pendidikan dan mobilitas sosial",
        "axial_category": "Pembangunan sumber daya manusia",
        "selective_theme": "Pembangunan manusia dan keadilan sosial",
        "keywords": [
            "sekolah", "pendidikan", "murid", "guru", "kampus", "universitas",
            "education", "school", "schools", "students", "teachers", "university", "universities"
        ],
        "description": "Mengacu pada pendidikan, sekolah rakyat, guru, siswa, dan peluang mobilitas sosial."
    },
    {
        "open_code_id": "OC005",
        "open_code_label": "Kesehatan, gizi, dan perlindungan sosial",
        "axial_category": "Pembangunan sumber daya manusia",
        "selective_theme": "Pembangunan manusia dan keadilan sosial",
        "keywords": [
            "makan bergizi", "mbg", "gizi", "kesehatan", "anak", "ibu hamil", "lansia",
            "meals", "nutrition", "nutritious", "medical", "health", "children", "pregnant", "elderly"
        ],
        "description": "Mengacu pada program makan bergizi, kesehatan, anak, ibu hamil, dan lansia."
    },
    {
        "open_code_id": "OC006",
        "open_code_label": "Anti-korupsi dan penegakan hukum",
        "axial_category": "Tata kelola pemerintahan dan rule of law",
        "selective_theme": "Reformasi tata kelola dan penegakan hukum",
        "keywords": [
            "korupsi", "hukum", "ilegal", "tambang ilegal", "disita", "markup", "penyelewengan",
            "corruption", "law", "illegal", "confiscated", "rule of law", "licenses", "violating"
        ],
        "description": "Mengacu pada pemberantasan korupsi, penyitaan aset ilegal, dan penegakan hukum."
    },
    {
        "open_code_id": "OC007",
        "open_code_label": "Reformasi birokrasi dan efisiensi negara",
        "axial_category": "Tata kelola pemerintahan dan rule of law",
        "selective_theme": "Reformasi tata kelola dan penegakan hukum",
        "keywords": [
            "regulasi", "efisiensi", "birokrasi", "anggaran", "kabinet", "pemerintah",
            "efficiency", "efficient", "budget", "bureaucracy", "governance", "government", "regulations"
        ],
        "description": "Mengacu pada efisiensi anggaran, penyederhanaan regulasi, dan tata kelola pemerintahan."
    },
    {
        "open_code_id": "OC008",
        "open_code_label": "Diplomasi multilateral dan kerja sama internasional",
        "axial_category": "Diplomasi dan tatanan global",
        "selective_theme": "Diplomasi, perdamaian, dan keadilan global",
        "keywords": [
            "brics", "pbb", "united nations", "wef", "multilateral", "multilateralism",
            "cooperation", "collaboration", "peace", "stability", "international"
        ],
        "description": "Mengacu pada forum internasional, kerja sama multilateral, perdamaian, dan stabilitas global."
    },
    {
        "open_code_id": "OC009",
        "open_code_label": "Palestina, Gaza, dan keadilan global",
        "axial_category": "Diplomasi dan tatanan global",
        "selective_theme": "Diplomasi, perdamaian, dan keadilan global",
        "keywords": [
            "palestina", "gaza", "israel", "two state", "two-state", "genocide",
            "palestine", "palestinians", "justice", "humanity", "human family"
        ],
        "description": "Mengacu pada isu Palestina, Gaza, solusi dua negara, dan keadilan kemanusiaan global."
    },
    {
        "open_code_id": "OC010",
        "open_code_label": "Investasi, industrialisasi, dan pertumbuhan ekonomi",
        "axial_category": "Transformasi ekonomi dan pembangunan produktif",
        "selective_theme": "Transformasi ekonomi dan pembangunan nasional",
        "keywords": [
            "investasi", "hilirisasi", "industrialisasi", "pertumbuhan", "danantara", "ekonomi",
            "investment", "industrialize", "industrialization", "growth", "capital", "downstream"
        ],
        "description": "Mengacu pada pertumbuhan ekonomi, investasi, Danantara, hilirisasi, dan industrialisasi."
    },
    {
        "open_code_id": "OC011",
        "open_code_label": "Koperasi, desa, dan ekonomi akar rumput",
        "axial_category": "Transformasi ekonomi dan pembangunan produktif",
        "selective_theme": "Transformasi ekonomi dan pembangunan nasional",
        "keywords": [
            "koperasi", "desa", "nelayan", "gudang", "cold storage", "pasar", "ekonomi bawah",
            "cooperatives", "village", "villages", "fishermen", "warehouses", "mini markets"
        ],
        "description": "Mengacu pada koperasi, desa nelayan, gudang, cold storage, dan ekonomi lokal."
    },
    {
        "open_code_id": "OC012",
        "open_code_label": "Lingkungan, iklim, dan keberlanjutan",
        "axial_category": "Ketahanan lingkungan dan keberlanjutan",
        "selective_theme": "Keberlanjutan lingkungan dan sumber daya",
        "keywords": [
            "iklim", "lingkungan", "emisi", "hutan", "reforestasi", "laut", "sampah",
            "climate", "emission", "emissions", "forest", "reforest", "sea level", "renewables", "waste"
        ],
        "description": "Mengacu pada perubahan iklim, emisi, hutan, laut, energi terbarukan, dan keberlanjutan."
    },
    {
        "open_code_id": "OC013",
        "open_code_label": "Persatuan nasional dan kepercayaan diri bangsa",
        "axial_category": "Identitas nasional dan kepemimpinan",
        "selective_theme": "Kedaulatan dan kemandirian nasional",
        "keywords": [
            "persatuan", "bangsa", "merah putih", "mandiri", "berdiri di atas kaki", "percaya diri",
            "unity", "nation", "national", "confidence", "sovereignty", "independence"
        ],
        "description": "Mengacu pada persatuan, kemandirian, nasionalisme, dan kepercayaan diri bangsa."
    }
]

manual_coding_codebook_df = pd.DataFrame(codebook_records)
manual_coding_codebook_df["keywords_json"] = manual_coding_codebook_df["keywords"].apply(
    lambda value: json.dumps(value, ensure_ascii=False)
)
manual_coding_codebook_df = manual_coding_codebook_df.drop(columns=["keywords"])

display(manual_coding_codebook_df)
print(f"Jumlah open code awal: {len(manual_coding_codebook_df)}")
print(f"Jumlah axial category: {manual_coding_codebook_df['axial_category'].nunique()}")
print(f"Jumlah selective theme: {manual_coding_codebook_df['selective_theme'].nunique()}")

,open_code_id,open_code_label,axial_category,selective_theme,description,keywords_json
0,OC001,Kedaulatan pangan dan swasembada,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,"Mengacu pada narasi kemandirian pangan, produk...","[""swasembada"", ""pangan"", ""beras"", ""jagung"", ""p..."
1,OC002,Kedaulatan energi dan pengelolaan sumber daya,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,"Mengacu pada kemandirian energi, Pertamina, BB...","[""energi"", ""pertamina"", ""biodiesel"", ""solar"", ..."
2,OC003,Pemerataan kesejahteraan dan pengentasan kemis...,Kesejahteraan sosial dan pemerataan pembangunan,Pembangunan manusia dan keadilan sosial,Mengacu pada upaya peningkatan kesejahteraan r...,"[""kemiskinan"", ""miskin"", ""kesejahteraan"", ""rak..."
3,OC004,Pendidikan dan mobilitas sosial,Pembangunan sumber daya manusia,Pembangunan manusia dan keadilan sosial,"Mengacu pada pendidikan, sekolah rakyat, guru,...","[""sekolah"", ""pendidikan"", ""murid"", ""guru"", ""ka..."
4,OC005,"Kesehatan, gizi, dan perlindungan sosial",Pembangunan sumber daya manusia,Pembangunan manusia dan keadilan sosial,"Mengacu pada program makan bergizi, kesehatan,...","[""makan bergizi"", ""mbg"", ""gizi"", ""kesehatan"", ..."
5,OC006,Anti-korupsi dan penegakan hukum,Tata kelola pemerintahan dan rule of law,Reformasi tata kelola dan penegakan hukum,"Mengacu pada pemberantasan korupsi, penyitaan ...","[""korupsi"", ""hukum"", ""ilegal"", ""tambang ilegal..."
6,OC007,Reformasi birokrasi dan efisiensi negara,Tata kelola pemerintahan dan rule of law,Reformasi tata kelola dan penegakan hukum,"Mengacu pada efisiensi anggaran, penyederhanaa...","[""regulasi"", ""efisiensi"", ""birokrasi"", ""anggar..."
7,OC008,Diplomasi multilateral dan kerja sama internas...,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global","Mengacu pada forum internasional, kerja sama m...","[""brics"", ""pbb"", ""united nations"", ""wef"", ""mul..."
8,OC009,"Palestina, Gaza, dan keadilan global",Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global","Mengacu pada isu Palestina, Gaza, solusi dua n...","[""palestina"", ""gaza"", ""israel"", ""two state"", ""..."
9,OC010,"Investasi, industrialisasi, dan pertumbuhan ek...",Transformasi ekonomi dan pembangunan produktif,Transformasi ekonomi dan pembangunan nasional,"Mengacu pada pertumbuhan ekonomi, investasi, D...","[""investasi"", ""hilirisasi"", ""industrialisasi"",..."


Jumlah open code awal: 13
Jumlah axial category: 8
Jumlah selective theme: 6


## 7. Fungsi Rule-Assisted Coding

Fungsi pada cell ini memberikan saran kode awal berdasarkan kemunculan keyword.

Prinsip validitas:

1. Saran dibuat secara transparan.
2. Keyword yang memicu saran disimpan sebagai evidence.
3. Saran tidak otomatis dianggap sebagai hasil final.
4. Chunk yang tidak memiliki kecocokan keyword diberi status `NEEDS_MANUAL_REVIEW`.

In [22]:
# ============================================================
# Helper Functions Rule-Assisted Coding
# ============================================================

def normalize_for_matching(text):
    """
    Normalisasi sederhana untuk pencocokan keyword.
    """
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = text.replace("’", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def count_keyword_occurrences(text, keyword):
    """
    Menghitung kemunculan keyword dalam teks.

    Untuk keyword satu kata digunakan batas kata.
    Untuk keyword frasa digunakan pencarian frasa.
    """
    text = normalize_for_matching(text)
    keyword = normalize_for_matching(keyword)

    if not keyword:
        return 0

    if " " in keyword or "-" in keyword:
        pattern = re.escape(keyword)
    else:
        pattern = r"(?<!\w)" + re.escape(keyword) + r"(?!\w)"

    return len(re.findall(pattern, text, flags=re.IGNORECASE))


def suggest_codes_for_chunk(chunk_text, codebook_df, top_n=3):
    """
    Memberikan saran open code untuk satu chunk berdasarkan keyword.
    """
    suggestions = []

    for _, code_row in codebook_df.iterrows():
        keywords = json.loads(code_row["keywords_json"])

        matched_keywords = []
        total_hits = 0

        for keyword in keywords:
            hit_count = count_keyword_occurrences(chunk_text, keyword)
            if hit_count > 0:
                matched_keywords.append({
                    "keyword": keyword,
                    "hit_count": hit_count
                })
                total_hits += hit_count

        if total_hits > 0:
            suggestions.append({
                "open_code_id": code_row["open_code_id"],
                "open_code_label": code_row["open_code_label"],
                "axial_category": code_row["axial_category"],
                "selective_theme": code_row["selective_theme"],
                "total_hits": total_hits,
                "unique_keyword_hits": len(matched_keywords),
                "matched_keywords": matched_keywords
            })

    suggestions = sorted(
        suggestions,
        key=lambda item: (
            item["unique_keyword_hits"],
            item["total_hits"],
            item["open_code_id"]
        ),
        reverse=True
    )

    return suggestions[:top_n]


def infer_suggestion_confidence(suggestions):
    """
    Menentukan level confidence sederhana berdasarkan evidence keyword.
    """
    if not suggestions:
        return "NEEDS_MANUAL_REVIEW"

    top = suggestions[0]

    if top["unique_keyword_hits"] >= 3 or top["total_hits"] >= 6:
        return "HIGH"

    if top["unique_keyword_hits"] >= 2 or top["total_hits"] >= 3:
        return "MEDIUM"

    return "LOW"


def format_matched_keywords(suggestions):
    """
    Merangkum evidence keyword agar mudah dibaca pada CSV.
    """
    if not suggestions:
        return ""

    parts = []

    for suggestion in suggestions:
        keywords = [
            f"{item['keyword']}({item['hit_count']})"
            for item in suggestion["matched_keywords"]
        ]
        parts.append(
            f"{suggestion['open_code_id']}={', '.join(keywords)}"
        )

    return " | ".join(parts)


def create_suggestion_json(suggestions):
    """
    Menyimpan detail suggestion sebagai JSON.
    """
    return json.dumps(suggestions, ensure_ascii=False)


print("Helper functions rule-assisted coding berhasil dibuat.")

Helper functions rule-assisted coding berhasil dibuat.


## 8. Membuat Manual Coding Template dan Rule-Assisted Draft

Cell ini menghasilkan dua file:

1. `manual_coding_template.csv`  
   Template untuk proses coding manual.

2. `manual_coding_rule_assisted_draft.csv`  
   Draft yang sudah memiliki saran kode berbasis keyword.

Kolom manual sengaja dikosongkan agar dapat diisi dan direview terlebih dahulu.

In [23]:
# ============================================================
# Generate Manual Coding Template
# ============================================================

metadata_columns_optional = [
    "speech_title_from_filename",
    "forum_scope_inferred",
    "event_date",
    "language_estimate",
    "source_url",
    "source_domain",
    "source_validation_status",
    "chunk_quality_flags"
]

manual_records = []

for _, row in chunks_df.iterrows():
    chunk_text = row["chunk_text"]
    suggestions = suggest_codes_for_chunk(
        chunk_text=chunk_text,
        codebook_df=manual_coding_codebook_df,
        top_n=3
    )

    suggestion_confidence = infer_suggestion_confidence(suggestions)

    suggested_open_codes = ["", "", ""]
    suggested_axial_categories = ["", "", ""]
    suggested_selective_themes = ["", "", ""]

    for idx, suggestion in enumerate(suggestions[:3]):
        suggested_open_codes[idx] = suggestion["open_code_label"]
        suggested_axial_categories[idx] = suggestion["axial_category"]
        suggested_selective_themes[idx] = suggestion["selective_theme"]

    record = {
        "chunk_id": row["chunk_id"],
        "speech_id": row["speech_id"],
        "file_name": row["file_name"],
        "chunk_order": row["chunk_order"],
        "chunk_text": chunk_text,
        "chunk_word_count": row["chunk_word_count"],
        "suggested_open_code_1": suggested_open_codes[0],
        "suggested_open_code_2": suggested_open_codes[1],
        "suggested_open_code_3": suggested_open_codes[2],
        "suggested_axial_category_1": suggested_axial_categories[0],
        "suggested_axial_category_2": suggested_axial_categories[1],
        "suggested_axial_category_3": suggested_axial_categories[2],
        "suggested_selective_theme_1": suggested_selective_themes[0],
        "suggested_selective_theme_2": suggested_selective_themes[1],
        "suggested_selective_theme_3": suggested_selective_themes[2],
        "suggestion_confidence": suggestion_confidence,
        "matched_keywords_summary": format_matched_keywords(suggestions),
        "suggestion_detail_json": create_suggestion_json(suggestions),
        "manual_open_code_1": "",
        "manual_open_code_2": "",
        "manual_open_code_3": "",
        "manual_axial_category": "",
        "manual_selective_theme": "",
        "coding_status": "UNREVIEWED",
        "coder_notes": "",
        "processed_stage04_at": datetime.now().isoformat(timespec="seconds")
    }

    for col in metadata_columns_optional:
        record[col] = safe_get(row, col, default_value=None)

    manual_records.append(record)


manual_coding_template_df = pd.DataFrame(manual_records)

MANUAL_TEMPLATE_COLUMNS = [
    "chunk_id",
    "speech_id",
    "file_name",
    "speech_title_from_filename",
    "forum_scope_inferred",
    "event_date",
    "language_estimate",
    "source_url",
    "source_domain",
    "source_validation_status",
    "chunk_order",
    "chunk_text",
    "chunk_word_count",
    "chunk_quality_flags",
    "suggested_open_code_1",
    "suggested_open_code_2",
    "suggested_open_code_3",
    "suggested_axial_category_1",
    "suggested_axial_category_2",
    "suggested_axial_category_3",
    "suggested_selective_theme_1",
    "suggested_selective_theme_2",
    "suggested_selective_theme_3",
    "suggestion_confidence",
    "matched_keywords_summary",
    "suggestion_detail_json",
    "manual_open_code_1",
    "manual_open_code_2",
    "manual_open_code_3",
    "manual_axial_category",
    "manual_selective_theme",
    "coding_status",
    "coder_notes",
    "processed_stage04_at"
]

require_columns(manual_coding_template_df, MANUAL_TEMPLATE_COLUMNS, "manual_coding_template_df")
manual_coding_template_df = manual_coding_template_df[MANUAL_TEMPLATE_COLUMNS].copy()

manual_coding_rule_assisted_draft_df = manual_coding_template_df.copy()

display(manual_coding_rule_assisted_draft_df.head())
print(f"Jumlah baris template manual coding: {len(manual_coding_template_df)}")

,chunk_id,speech_id,file_name,speech_title_from_filename,forum_scope_inferred,event_date,language_estimate,source_url,source_domain,source_validation_status,...,matched_keywords_summary,suggestion_detail_json,manual_open_code_1,manual_open_code_2,manual_open_code_3,manual_axial_category,manual_selective_theme,coding_status,coder_notes,processed_stage04_at
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,"OC008=brics(6), multilateralism(1), cooperatio...","[{""open_code_id"": ""OC008"", ""open_code_label"": ...",,,,,,UNREVIEWED,,2026-06-11T23:08:49
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,OC008=brics(2),"[{""open_code_id"": ""OC008"", ""open_code_label"": ...",,,,,,UNREVIEWED,,2026-06-11T23:08:49
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,"OC001=pangan(1), jagung(1), petani(1), pertani...","[{""open_code_id"": ""OC001"", ""open_code_label"": ...",,,,,,UNREVIEWED,,2026-06-11T23:08:49
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,"OC001=pangan(1), jagung(1), petani(2), pertani...","[{""open_code_id"": ""OC001"", ""open_code_label"": ...",,,,,,UNREVIEWED,,2026-06-11T23:08:49
4,SPCH_002_PANEN_RAYA_CHK_003,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,OC005=kesehatan(1),"[{""open_code_id"": ""OC005"", ""open_code_label"": ...",,,,,,UNREVIEWED,,2026-06-11T23:08:49


Jumlah baris template manual coding: 74


## 9. Validasi Template Manual Coding

Validasi dilakukan untuk memastikan:

1. Jumlah baris template sama dengan jumlah chunk.
2. Tidak ada `chunk_id` kosong.
3. `chunk_id` tetap unik.
4. Kolom manual tersedia.
5. Kolom suggestion tersedia.

In [24]:
# ============================================================
# Cell 09 — Validasi Template Manual Coding
# ============================================================

if len(manual_coding_template_df) != len(chunks_df):
    raise ValueError(
        f"Jumlah baris template tidak sama dengan jumlah chunk. "
        f"Template: {len(manual_coding_template_df)}, Chunk: {len(chunks_df)}"
    )

if manual_coding_template_df["chunk_id"].isna().any():
    raise ValueError("Terdapat chunk_id kosong pada manual_coding_template_df.")

if manual_coding_template_df["chunk_id"].duplicated().any():
    raise ValueError("Terdapat chunk_id duplikat pada manual_coding_template_df.")

required_manual_columns = [
    "manual_open_code_1",
    "manual_open_code_2",
    "manual_open_code_3",
    "manual_axial_category",
    "manual_selective_theme",
    "coding_status",
    "coder_notes"
]

require_columns(manual_coding_template_df, required_manual_columns, "manual_coding_template_df")

print("Validasi template manual coding berhasil.")

Validasi template manual coding berhasil.


## 10. Ringkasan Suggestion Manual Coding

Ringkasan ini hanya menggambarkan distribusi saran berbasis keyword.

Hasil ini belum boleh diperlakukan sebagai hasil final penelitian sebelum direview secara manual.

In [25]:
# ============================================================
# Manual Coding Suggestion Summary
# ============================================================

suggestion_summary_records = []

suggestion_summary_records.append({
    "metric": "total_chunks",
    "value": int(len(manual_coding_rule_assisted_draft_df)),
    "description": "Jumlah total chunk yang masuk ke template manual coding."
})

suggestion_summary_records.append({
    "metric": "chunks_with_suggestion",
    "value": int((manual_coding_rule_assisted_draft_df["suggestion_confidence"] != "NEEDS_MANUAL_REVIEW").sum()),
    "description": "Jumlah chunk yang memiliki minimal satu saran kode berbasis keyword."
})

suggestion_summary_records.append({
    "metric": "chunks_needing_manual_review",
    "value": int((manual_coding_rule_assisted_draft_df["suggestion_confidence"] == "NEEDS_MANUAL_REVIEW").sum()),
    "description": "Jumlah chunk yang tidak memiliki kecocokan keyword dan perlu direview manual."
})

suggestion_summary_records.append({
    "metric": "high_confidence_suggestions",
    "value": int((manual_coding_rule_assisted_draft_df["suggestion_confidence"] == "HIGH").sum()),
    "description": "Jumlah chunk dengan suggestion confidence HIGH."
})

suggestion_summary_records.append({
    "metric": "medium_confidence_suggestions",
    "value": int((manual_coding_rule_assisted_draft_df["suggestion_confidence"] == "MEDIUM").sum()),
    "description": "Jumlah chunk dengan suggestion confidence MEDIUM."
})

suggestion_summary_records.append({
    "metric": "low_confidence_suggestions",
    "value": int((manual_coding_rule_assisted_draft_df["suggestion_confidence"] == "LOW").sum()),
    "description": "Jumlah chunk dengan suggestion confidence LOW."
})

manual_coding_suggestion_summary_df = pd.DataFrame(suggestion_summary_records)

display(manual_coding_suggestion_summary_df)

print("Distribusi suggested_selective_theme_1:")
display(
    manual_coding_rule_assisted_draft_df["suggested_selective_theme_1"]
    .replace("", "NO_SUGGESTION")
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={
        "index": "suggested_selective_theme_1",
        "suggested_selective_theme_1": "chunk_count"
    })
)

print("Distribusi suggestion_confidence:")
display(
    manual_coding_rule_assisted_draft_df["suggestion_confidence"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={
        "index": "suggestion_confidence",
        "suggestion_confidence": "chunk_count"
    })
)

,metric,value,description
0,total_chunks,74,Jumlah total chunk yang masuk ke template manu...
1,chunks_with_suggestion,74,Jumlah chunk yang memiliki minimal satu saran ...
2,chunks_needing_manual_review,0,Jumlah chunk yang tidak memiliki kecocokan key...
3,high_confidence_suggestions,49,Jumlah chunk dengan suggestion confidence HIGH.
4,medium_confidence_suggestions,17,Jumlah chunk dengan suggestion confidence MEDIUM.
5,low_confidence_suggestions,8,Jumlah chunk dengan suggestion confidence LOW.


Distribusi suggested_selective_theme_1:


,chunk_count,count
0,Pembangunan manusia dan keadilan sosial,27
1,Kedaulatan dan kemandirian nasional,22
2,"Diplomasi, perdamaian, dan keadilan global",12
3,Reformasi tata kelola dan penegakan hukum,6
4,Transformasi ekonomi dan pembangunan nasional,5
5,Keberlanjutan lingkungan dan sumber daya,2


Distribusi suggestion_confidence:


,chunk_count,count
0,HIGH,49
1,MEDIUM,17
2,LOW,8


## 11. Quality Report Manual Coding

Quality report digunakan untuk mengecek kesiapan data sebelum direview secara manual.

In [26]:
# ============================================================
# Manual Coding Quality Report
# ============================================================

quality_records = []

quality_records.append({
    "metric": "input_chunk_count",
    "value": int(len(chunks_df)),
    "description": "Jumlah chunk input dari Tahap 03."
})

quality_records.append({
    "metric": "manual_template_row_count",
    "value": int(len(manual_coding_template_df)),
    "description": "Jumlah baris pada template manual coding."
})

quality_records.append({
    "metric": "unique_chunk_id_count",
    "value": int(manual_coding_template_df["chunk_id"].nunique()),
    "description": "Jumlah chunk_id unik pada template."
})

quality_records.append({
    "metric": "unreviewed_row_count",
    "value": int((manual_coding_template_df["coding_status"] == "UNREVIEWED").sum()),
    "description": "Jumlah baris yang belum direview manual."
})

quality_records.append({
    "metric": "manual_open_code_filled_count",
    "value": int((manual_coding_template_df["manual_open_code_1"].astype(str).str.strip() != "").sum()),
    "description": "Jumlah baris yang sudah memiliki manual_open_code_1. Pada template awal biasanya 0."
})

quality_records.append({
    "metric": "source_speech_count",
    "value": int(manual_coding_template_df["speech_id"].nunique()),
    "description": "Jumlah pidato unik pada template manual coding."
})

manual_coding_quality_report_df = pd.DataFrame(quality_records)

display(manual_coding_quality_report_df)

,metric,value,description
0,input_chunk_count,74,Jumlah chunk input dari Tahap 03.
1,manual_template_row_count,74,Jumlah baris pada template manual coding.
2,unique_chunk_id_count,74,Jumlah chunk_id unik pada template.
3,unreviewed_row_count,74,Jumlah baris yang belum direview manual.
4,manual_open_code_filled_count,0,Jumlah baris yang sudah memiliki manual_open_c...
5,source_speech_count,6,Jumlah pidato unik pada template manual coding.


## 12. Optional — Membaca Hasil Coding Manual yang Sudah Direview

Setelah template direview secara manual, simpan hasilnya sebagai:

```text
data/interim/manual_coding_reviewed.csv
```

Notebook akan otomatis membaca file tersebut jika tersedia.

Syarat minimal file reviewed:

- `chunk_id`
- `manual_open_code_1`
- `manual_axial_category`
- `manual_selective_theme`
- `coding_status`

Jika file belum tersedia, tahap ini akan dilewati tanpa error.

In [27]:
# ============================================================
# Optional Load Manual Coding Reviewed
# ============================================================

MANUAL_CODING_REVIEWED_PATH = INTERIM_DIR / "manual_coding_reviewed.csv"

manual_coding_final_df = None

if MANUAL_CODING_REVIEWED_PATH.exists():
    print(f"File manual coding reviewed ditemukan: {MANUAL_CODING_REVIEWED_PATH}")

    reviewed_df = pd.read_csv(MANUAL_CODING_REVIEWED_PATH)

    reviewed_required_columns = [
        "chunk_id",
        "manual_open_code_1",
        "manual_axial_category",
        "manual_selective_theme",
        "coding_status"
    ]

    require_columns(reviewed_df, reviewed_required_columns, "reviewed_df")

    if reviewed_df["chunk_id"].duplicated().any():
        raise ValueError("Terdapat chunk_id duplikat pada manual_coding_reviewed.csv.")

    missing_reviewed_chunk_ids = set(chunks_df["chunk_id"]) - set(reviewed_df["chunk_id"])
    extra_reviewed_chunk_ids = set(reviewed_df["chunk_id"]) - set(chunks_df["chunk_id"])

    if missing_reviewed_chunk_ids:
        raise ValueError(
            f"Terdapat chunk_id dari data chunk yang belum ada pada file reviewed: "
            f"{list(missing_reviewed_chunk_ids)[:10]}"
        )

    if extra_reviewed_chunk_ids:
        raise ValueError(
            f"Terdapat chunk_id pada file reviewed yang tidak ada pada data chunk: "
            f"{list(extra_reviewed_chunk_ids)[:10]}"
        )

    manual_coding_final_df = reviewed_df.copy()

    print("File manual coding reviewed berhasil divalidasi.")
    display(manual_coding_final_df.head())

else:
    print(
        "File manual_coding_reviewed.csv belum tersedia. "
        "Tahap final manual coding dilewati. "
        "Gunakan manual_coding_template.csv untuk proses review manual."
    )

File manual coding reviewed ditemukan: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\interim\manual_coding_reviewed.csv
File manual coding reviewed berhasil divalidasi.


,chunk_id,speech_id,file_name,speech_title_from_filename,forum_scope_inferred,event_date,language_estimate,source_url,source_domain,source_validation_status,...,matched_keywords_summary,suggestion_detail_json,manual_open_code_1,manual_open_code_2,manual_open_code_3,manual_axial_category,manual_selective_theme,coding_status,coder_notes,processed_stage04_at
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,9/8/2025,en,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,"OC008=brics(6), multilateralism(1), cooperatio...","[{""open_code_id"": ""OC008"", ""open_code_label"": ...",Diplomasi multilateral dan kerja sama internas...,Anti-korupsi dan penegakan hukum,NaN,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global",REVIEWED,NaN,2026-06-11T22:11:05
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,9/8/2025,en,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,OC008=brics(2),"[{""open_code_id"": ""OC008"", ""open_code_label"": ...",Diplomasi multilateral dan kerja sama internas...,NaN,NaN,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global",REVIEWED,NaN,2026-06-11T22:11:05
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,1/7/2026,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,"OC001=pangan(1), jagung(1), petani(1), pertani...","[{""open_code_id"": ""OC001"", ""open_code_label"": ...",Kedaulatan pangan dan swasembada,Pemerataan kesejahteraan dan pengentasan kemis...,NaN,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,REVIEWED,NaN,2026-06-11T22:11:05
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,1/7/2026,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,"OC001=pangan(1), jagung(1), petani(2), pertani...","[{""open_code_id"": ""OC001"", ""open_code_label"": ...",Kedaulatan pangan dan swasembada,Pemerataan kesejahteraan dan pengentasan kemis...,NaN,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,REVIEWED,NaN,2026-06-11T22:11:05
4,SPCH_002_PANEN_RAYA_CHK_003,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,1/7/2026,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,OC005=kesehatan(1),"[{""open_code_id"": ""OC005"", ""open_code_label"": ...","Kesehatan, gizi, dan perlindungan sosial",NaN,NaN,Pembangunan sumber daya manusia,Pembangunan manusia dan keadilan sosial,EXCLUDED,NaN,2026-06-11T22:11:05


## 13. Optional — Ringkasan Final Manual Coding

Cell ini hanya berjalan jika file `manual_coding_reviewed.csv` sudah tersedia.

In [28]:
# ============================================================
# Optional Final Manual Coding Summary
# ============================================================

manual_coding_final_theme_distribution_df = pd.DataFrame()
manual_coding_final_axial_distribution_df = pd.DataFrame()
manual_coding_final_code_distribution_df = pd.DataFrame()

if manual_coding_final_df is not None:
    manual_coding_final_theme_distribution_df = (
        manual_coding_final_df["manual_selective_theme"]
        .fillna("MISSING")
        .replace("", "MISSING")
        .value_counts()
        .reset_index()
        .rename(columns={
            "index": "manual_selective_theme",
            "manual_selective_theme": "chunk_count"
        })
    )

    manual_coding_final_axial_distribution_df = (
        manual_coding_final_df["manual_axial_category"]
        .fillna("MISSING")
        .replace("", "MISSING")
        .value_counts()
        .reset_index()
        .rename(columns={
            "index": "manual_axial_category",
            "manual_axial_category": "chunk_count"
        })
    )

    manual_coding_final_code_distribution_df = (
        manual_coding_final_df["manual_open_code_1"]
        .fillna("MISSING")
        .replace("", "MISSING")
        .value_counts()
        .reset_index()
        .rename(columns={
            "index": "manual_open_code_1",
            "manual_open_code_1": "chunk_count"
        })
    )

    print("Distribusi Selective Theme Final:")
    display(manual_coding_final_theme_distribution_df)

    print("Distribusi Axial Category Final:")
    display(manual_coding_final_axial_distribution_df)

    print("Distribusi Open Code Final:")
    display(manual_coding_final_code_distribution_df)

else:
    print("Ringkasan final manual coding dilewati karena manual_coding_reviewed.csv belum tersedia.")

Distribusi Selective Theme Final:


,chunk_count,count
0,Pembangunan manusia dan keadilan sosial,27
1,Kedaulatan dan kemandirian nasional,22
2,"Diplomasi, perdamaian, dan keadilan global",12
3,Reformasi tata kelola dan penegakan hukum,6
4,Transformasi ekonomi dan pembangunan nasional,5
5,Keberlanjutan lingkungan dan sumber daya,2


Distribusi Axial Category Final:


,chunk_count,count
0,Pembangunan sumber daya manusia,19
1,Ketahanan nasional dan kedaulatan strategis,16
2,Diplomasi dan tatanan global,12
3,Kesejahteraan sosial dan pemerataan pembangunan,8
4,Identitas nasional dan kepemimpinan,6
5,Tata kelola pemerintahan dan rule of law,6
6,Transformasi ekonomi dan pembangunan produktif,5
7,Ketahanan lingkungan dan keberlanjutan,2


Distribusi Open Code Final:


,chunk_count,count
0,Kedaulatan pangan dan swasembada,12
1,"Kesehatan, gizi, dan perlindungan sosial",11
2,Diplomasi multilateral dan kerja sama internas...,9
3,Pemerataan kesejahteraan dan pengentasan kemis...,8
4,Pendidikan dan mobilitas sosial,8
5,Persatuan nasional dan kepercayaan diri bangsa,6
6,Anti-korupsi dan penegakan hukum,4
7,Kedaulatan energi dan pengelolaan sumber daya,4
8,"Investasi, industrialisasi, dan pertumbuhan ek...",3
9,"Palestina, Gaza, dan keadilan global",3


## 14. Menyimpan Output Tahap 04

Output utama tahap ini disimpan ke:

```text
data/interim/
reports/tables/
```

Jika file reviewed tersedia, output final juga akan disimpan ke:

```text
data/processed/manual_coding_final.csv
reports/tables/manual_coding_final_*.csv
```

In [29]:
# ============================================================
# Save Output Tahap 04
# ============================================================

manual_coding_codebook_path = INTERIM_DIR / "manual_coding_codebook.csv"
manual_coding_template_path = INTERIM_DIR / "manual_coding_template.csv"
manual_coding_rule_assisted_draft_path = INTERIM_DIR / "manual_coding_rule_assisted_draft.csv"

manual_coding_suggestion_summary_path = REPORT_TABLE_DIR / "manual_coding_suggestion_summary.csv"
manual_coding_quality_report_path = REPORT_TABLE_DIR / "manual_coding_quality_report.csv"
stage04_output_manifest_path = REPORT_TABLE_DIR / "stage04_output_manifest.json"

manual_coding_codebook_df.to_csv(manual_coding_codebook_path, index=False, encoding="utf-8-sig")
manual_coding_template_df.to_csv(manual_coding_template_path, index=False, encoding="utf-8-sig")
manual_coding_rule_assisted_draft_df.to_csv(manual_coding_rule_assisted_draft_path, index=False, encoding="utf-8-sig")
manual_coding_suggestion_summary_df.to_csv(manual_coding_suggestion_summary_path, index=False, encoding="utf-8-sig")
manual_coding_quality_report_df.to_csv(manual_coding_quality_report_path, index=False, encoding="utf-8-sig")

final_outputs = {}

if manual_coding_final_df is not None:
    manual_coding_final_path = PROCESSED_DIR / "manual_coding_final.csv"
    manual_coding_final_theme_distribution_path = REPORT_TABLE_DIR / "manual_coding_final_theme_distribution.csv"
    manual_coding_final_axial_distribution_path = REPORT_TABLE_DIR / "manual_coding_final_axial_distribution.csv"
    manual_coding_final_code_distribution_path = REPORT_TABLE_DIR / "manual_coding_final_code_distribution.csv"

    manual_coding_final_df.to_csv(manual_coding_final_path, index=False, encoding="utf-8-sig")
    manual_coding_final_theme_distribution_df.to_csv(
        manual_coding_final_theme_distribution_path,
        index=False,
        encoding="utf-8-sig"
    )
    manual_coding_final_axial_distribution_df.to_csv(
        manual_coding_final_axial_distribution_path,
        index=False,
        encoding="utf-8-sig"
    )
    manual_coding_final_code_distribution_df.to_csv(
        manual_coding_final_code_distribution_path,
        index=False,
        encoding="utf-8-sig"
    )

    final_outputs = {
        "manual_coding_final": str(manual_coding_final_path),
        "manual_coding_final_theme_distribution": str(manual_coding_final_theme_distribution_path),
        "manual_coding_final_axial_distribution": str(manual_coding_final_axial_distribution_path),
        "manual_coding_final_code_distribution": str(manual_coding_final_code_distribution_path)
    }

stage04_output_manifest = {
    "stage": "04_manual_coding_open_axial_selective",
    "input_file": str(INPUT_CHUNKS_MASTER),
    "outputs": {
        "manual_coding_codebook": str(manual_coding_codebook_path),
        "manual_coding_template": str(manual_coding_template_path),
        "manual_coding_rule_assisted_draft": str(manual_coding_rule_assisted_draft_path),
        "manual_coding_suggestion_summary": str(manual_coding_suggestion_summary_path),
        "manual_coding_quality_report": str(manual_coding_quality_report_path),
        **final_outputs
    },
    "input_chunk_count": int(len(chunks_df)),
    "source_speech_count": int(chunks_df["speech_id"].nunique()),
    "codebook_open_code_count": int(len(manual_coding_codebook_df)),
    "manual_reviewed_file_found": bool(manual_coding_final_df is not None),
    "created_at": datetime.now().isoformat(timespec="seconds")
}

stage04_output_manifest_path.write_text(
    json.dumps(stage04_output_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Output Tahap 04 berhasil disimpan:")
print(f"1. {manual_coding_codebook_path}")
print(f"2. {manual_coding_template_path}")
print(f"3. {manual_coding_rule_assisted_draft_path}")
print(f"4. {manual_coding_suggestion_summary_path}")
print(f"5. {manual_coding_quality_report_path}")
print(f"6. {stage04_output_manifest_path}")

if final_outputs:
    print("Output final manual coding juga berhasil disimpan:")
    for key, value in final_outputs.items():
        print(f"- {key}: {value}")

Output Tahap 04 berhasil disimpan:
1. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\interim\manual_coding_codebook.csv
2. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\interim\manual_coding_template.csv
3. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\interim\manual_coding_rule_assisted_draft.csv
4. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\manual_coding_suggestion_summary.csv
5. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\manual_coding_quality_report.csv
6. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\stage04_output_manifest.json
Output final manual coding juga berhasil disimpan:
- manual_coding_final: D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\manual_coding_final.csv
- manual_coding_fina

In [30]:
# ============================================================
# Auto-fill Draft Manual Coding dari Suggested Coding
# ============================================================

manual_review_draft_df = manual_coding_template_df.copy()

# Auto-fill hanya untuk baris yang memiliki suggestion
has_suggestion = manual_review_draft_df["suggestion_confidence"].isin(["HIGH", "MEDIUM", "LOW"])

manual_review_draft_df.loc[has_suggestion, "manual_open_code_1"] = (
    manual_review_draft_df.loc[has_suggestion, "suggested_open_code_1"]
)

manual_review_draft_df.loc[has_suggestion, "manual_open_code_2"] = (
    manual_review_draft_df.loc[has_suggestion, "suggested_open_code_2"]
)

manual_review_draft_df.loc[has_suggestion, "manual_open_code_3"] = (
    manual_review_draft_df.loc[has_suggestion, "suggested_open_code_3"]
)

manual_review_draft_df.loc[has_suggestion, "manual_axial_category"] = (
    manual_review_draft_df.loc[has_suggestion, "suggested_axial_category_1"]
)

manual_review_draft_df.loc[has_suggestion, "manual_selective_theme"] = (
    manual_review_draft_df.loc[has_suggestion, "suggested_selective_theme_1"]
)

# Status awal: belum final, tetap perlu dicek manusia
manual_review_draft_df.loc[
    manual_review_draft_df["suggestion_confidence"].isin(["HIGH", "MEDIUM"]),
    "coding_status"
] = "REVIEWED"

manual_review_draft_df.loc[
    manual_review_draft_df["suggestion_confidence"].isin(["LOW", "NEEDS_MANUAL_REVIEW"]),
    "coding_status"
] = "NEEDS_DISCUSSION"

manual_review_draft_df["coder_notes"] = manual_review_draft_df["coder_notes"].fillna("")

# Simpan sebagai draft review
manual_review_draft_path = INTERIM_DIR / "manual_coding_reviewed_draft.csv"

manual_review_draft_df.to_csv(
    manual_review_draft_path,
    index=False,
    encoding="utf-8-sig"
)

print("Draft manual coding berhasil dibuat:")
print(manual_review_draft_path)

display(
    manual_review_draft_df[
        [
            "chunk_id",
            "speech_id",
            "chunk_order",
            "suggestion_confidence",
            "manual_open_code_1",
            "manual_axial_category",
            "manual_selective_theme",
            "coding_status"
        ]
    ].head(20)
)

Draft manual coding berhasil dibuat:
D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\interim\manual_coding_reviewed_draft.csv


,chunk_id,speech_id,chunk_order,suggestion_confidence,manual_open_code_1,manual_axial_category,manual_selective_theme,coding_status
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,1,HIGH,Diplomasi multilateral dan kerja sama internas...,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global",REVIEWED
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,2,LOW,Diplomasi multilateral dan kerja sama internas...,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global",NEEDS_DISCUSSION
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,1,HIGH,Kedaulatan pangan dan swasembada,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,REVIEWED
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,2,HIGH,Kedaulatan pangan dan swasembada,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,REVIEWED
4,SPCH_002_PANEN_RAYA_CHK_003,SPCH_002_PANEN_RAYA,3,LOW,"Kesehatan, gizi, dan perlindungan sosial",Pembangunan sumber daya manusia,Pembangunan manusia dan keadilan sosial,NEEDS_DISCUSSION
5,SPCH_002_PANEN_RAYA_CHK_004,SPCH_002_PANEN_RAYA,4,HIGH,Persatuan nasional dan kepercayaan diri bangsa,Identitas nasional dan kepemimpinan,Kedaulatan dan kemandirian nasional,REVIEWED
6,SPCH_002_PANEN_RAYA_CHK_005,SPCH_002_PANEN_RAYA,5,MEDIUM,Pemerataan kesejahteraan dan pengentasan kemis...,Kesejahteraan sosial dan pemerataan pembangunan,Pembangunan manusia dan keadilan sosial,REVIEWED
7,SPCH_002_PANEN_RAYA_CHK_006,SPCH_002_PANEN_RAYA,6,MEDIUM,"Koperasi, desa, dan ekonomi akar rumput",Transformasi ekonomi dan pembangunan produktif,Transformasi ekonomi dan pembangunan nasional,REVIEWED
8,SPCH_002_PANEN_RAYA_CHK_007,SPCH_002_PANEN_RAYA,7,MEDIUM,Pemerataan kesejahteraan dan pengentasan kemis...,Kesejahteraan sosial dan pemerataan pembangunan,Pembangunan manusia dan keadilan sosial,REVIEWED
9,SPCH_002_PANEN_RAYA_CHK_008,SPCH_002_PANEN_RAYA,8,HIGH,Pemerataan kesejahteraan dan pengentasan kemis...,Kesejahteraan sosial dan pemerataan pembangunan,Pembangunan manusia dan keadilan sosial,REVIEWED


## 15. Preview Output Tahap 04

Preview ini digunakan untuk memastikan template dan saran coding sudah terbentuk dengan benar.

In [31]:
# ============================================================
# Preview Output Tahap 04
# ============================================================

preview_columns = [
    "chunk_id",
    "speech_id",
    "forum_scope_inferred",
    "chunk_order",
    "chunk_word_count",
    "suggested_open_code_1",
    "suggested_axial_category_1",
    "suggested_selective_theme_1",
    "suggestion_confidence",
    "manual_open_code_1",
    "manual_axial_category",
    "manual_selective_theme",
    "coding_status"
]

display(manual_coding_rule_assisted_draft_df[preview_columns].head(15))

print("Manual Coding Suggestion Summary:")
display(manual_coding_suggestion_summary_df)

print("Manual Coding Quality Report:")
display(manual_coding_quality_report_df)

,chunk_id,speech_id,forum_scope_inferred,chunk_order,chunk_word_count,suggested_open_code_1,suggested_axial_category_1,suggested_selective_theme_1,suggestion_confidence,manual_open_code_1,manual_axial_category,manual_selective_theme,coding_status
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,international,1,227,Diplomasi multilateral dan kerja sama internas...,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global",HIGH,,,,UNREVIEWED
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,international,2,42,Diplomasi multilateral dan kerja sama internas...,Diplomasi dan tatanan global,"Diplomasi, perdamaian, dan keadilan global",LOW,,,,UNREVIEWED
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,national,1,268,Kedaulatan pangan dan swasembada,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,HIGH,,,,UNREVIEWED
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,national,2,224,Kedaulatan pangan dan swasembada,Ketahanan nasional dan kedaulatan strategis,Kedaulatan dan kemandirian nasional,HIGH,,,,UNREVIEWED
4,SPCH_002_PANEN_RAYA_CHK_003,SPCH_002_PANEN_RAYA,national,3,242,"Kesehatan, gizi, dan perlindungan sosial",Pembangunan sumber daya manusia,Pembangunan manusia dan keadilan sosial,LOW,,,,UNREVIEWED
5,SPCH_002_PANEN_RAYA_CHK_004,SPCH_002_PANEN_RAYA,national,4,222,Persatuan nasional dan kepercayaan diri bangsa,Identitas nasional dan kepemimpinan,Kedaulatan dan kemandirian nasional,HIGH,,,,UNREVIEWED
6,SPCH_002_PANEN_RAYA_CHK_005,SPCH_002_PANEN_RAYA,national,5,233,Pemerataan kesejahteraan dan pengentasan kemis...,Kesejahteraan sosial dan pemerataan pembangunan,Pembangunan manusia dan keadilan sosial,MEDIUM,,,,UNREVIEWED
7,SPCH_002_PANEN_RAYA_CHK_006,SPCH_002_PANEN_RAYA,national,6,234,"Koperasi, desa, dan ekonomi akar rumput",Transformasi ekonomi dan pembangunan produktif,Transformasi ekonomi dan pembangunan nasional,MEDIUM,,,,UNREVIEWED
8,SPCH_002_PANEN_RAYA_CHK_007,SPCH_002_PANEN_RAYA,national,7,233,Pemerataan kesejahteraan dan pengentasan kemis...,Kesejahteraan sosial dan pemerataan pembangunan,Pembangunan manusia dan keadilan sosial,MEDIUM,,,,UNREVIEWED
9,SPCH_002_PANEN_RAYA_CHK_008,SPCH_002_PANEN_RAYA,national,8,221,Pemerataan kesejahteraan dan pengentasan kemis...,Kesejahteraan sosial dan pemerataan pembangunan,Pembangunan manusia dan keadilan sosial,HIGH,,,,UNREVIEWED


Manual Coding Suggestion Summary:


,metric,value,description
0,total_chunks,74,Jumlah total chunk yang masuk ke template manu...
1,chunks_with_suggestion,74,Jumlah chunk yang memiliki minimal satu saran ...
2,chunks_needing_manual_review,0,Jumlah chunk yang tidak memiliki kecocokan key...
3,high_confidence_suggestions,49,Jumlah chunk dengan suggestion confidence HIGH.
4,medium_confidence_suggestions,17,Jumlah chunk dengan suggestion confidence MEDIUM.
5,low_confidence_suggestions,8,Jumlah chunk dengan suggestion confidence LOW.


Manual Coding Quality Report:


,metric,value,description
0,input_chunk_count,74,Jumlah chunk input dari Tahap 03.
1,manual_template_row_count,74,Jumlah baris pada template manual coding.
2,unique_chunk_id_count,74,Jumlah chunk_id unik pada template.
3,unreviewed_row_count,74,Jumlah baris yang belum direview manual.
4,manual_open_code_filled_count,0,Jumlah baris yang sudah memiliki manual_open_c...
5,source_speech_count,6,Jumlah pidato unik pada template manual coding.


## 16. Panduan Review Manual Coding

Setelah notebook ini selesai dijalankan:

1. Buka file:
   ```text
   data/interim/manual_coding_template.csv
   ```

2. Review setiap chunk pada kolom:
   ```text
   chunk_text
   ```

3. Isi kolom manual:
   ```text
   manual_open_code_1
   manual_open_code_2
   manual_open_code_3
   manual_axial_category
   manual_selective_theme
   coding_status
   coder_notes
   ```

4. Gunakan `coding_status` berikut:
   ```text
   REVIEWED
   NEEDS_DISCUSSION
   EXCLUDED
   ```

5. Simpan hasil review sebagai:
   ```text
   data/interim/manual_coding_reviewed.csv
   ```

6. Jalankan ulang notebook ini dari awal untuk menghasilkan:
   ```text
   data/processed/manual_coding_final.csv
   reports/tables/manual_coding_final_theme_distribution.csv
   reports/tables/manual_coding_final_axial_distribution.csv
   reports/tables/manual_coding_final_code_distribution.csv
   ```

## Catatan Interpretasi

Saran kode berbasis keyword hanya digunakan sebagai alat bantu awal.  
Keputusan akhir coding harus berdasarkan pembacaan konteks chunk, bukan hanya kemunculan kata.